# Cold Mail Bot
> Automatically generates and sends personalized cold emails from LinkedIn posts using AI.

**Tech Stack:** Python, Groq AI (LLaMA 3.3 70B), Gmail API, Hunter.io, Selenium

**Website:** [cold-mail-frontend.vercel.app](https://cold-mail-frontend.vercel.app)  
**GitHub:** [cold-mail-bot](https://github.com/ShouryaGoel007/cold-mail-bot)


##  Step 1 — Install Dependencies

In [ ]:
!pip install selenium webdriver-manager groq google-auth google-auth-oauthlib google-api-python-client yagmail requests nest_asyncio

##  Step 2 — Configuration
Fill in your details here. This is the only cell you need to edit.

In [ ]:
# ============================================================
# FILL IN YOUR DETAILS HERE
# ============================================================

GROQ_API_KEY   = "your-groq-api-key"
HUNTER_API_KEY = "your-hunter-io-api-key"

CREDENTIALS_FILE = r"C:\Users\YourName\Desktop\cold_mail_bot\credentials_v3.json"
TOKEN_FILE       = r"C:\Users\YourName\Desktop\cold_mail_bot\token.pickle"

YOUR_PROFILE = """
My name is Shourya Goel.
I am a third year student pursuing B.Tech from IIT BHU in Chemical Engineering.
My top skills are Python, Excel, Machine Learning, Data Management, and Product Management.
I am looking for an internship.
"""


##  Step 3 — Imports

In [ ]:
import re
import os
import time
import base64
import pickle
import requests
from groq import Groq
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

print(" All imports successful!")


## Step 4 — LinkedIn Scraper
Opens your default browser (Brave/Chrome/Edge), logs into LinkedIn using your saved profile, and scrapes the post content.

In [ ]:
def get_brave_path():
    possible_paths = [
        r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe",
        r"C:\Program Files (x86)\BraveSoftware\Brave-Browser\Application\brave.exe",
        os.path.expanduser(r"~\AppData\Local\BraveSoftware\Brave-Browser\Application\brave.exe")
    ]
    for path in possible_paths:
        if os.path.exists(path):
            return path
    return None

def scrape_linkedin_post(url):
    brave_path = get_brave_path()
    options = Options()
    if brave_path:
        options.binary_location = brave_path
    brave_profile = os.path.expanduser(r"~\AppData\Local\BraveSoftware\Brave-Browser\User Data")
    options.add_argument(f"--user-data-dir={brave_profile}")
    options.add_argument("--profile-directory=Default")
    options.add_argument("--new-tab")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    print(" Opening LinkedIn post...")
    driver.get(url)
    time.sleep(5)

    if "login" in driver.current_url or "authwall" in driver.current_url:
        print(" LinkedIn asked you to log in.")
        input(" Please log in manually, then press ENTER here...")
        time.sleep(3)

    full_text = driver.find_element(By.TAG_NAME, "body").text
    driver.quit()

    emails_found = re.findall(r'[\w\.-]+@[\w\.-]+\.\w+', full_text)
    print(" Post scraped successfully!")
    print(f" Emails found: {emails_found if emails_found else 'None'}")

    return {
        "full_text": full_text,
        "email_in_post": emails_found[0] if emails_found else None
    }

print(" Scraper ready!")


##  Step 5 — Email Finder (Hunter.io)
Finds HR email automatically using their name and company domain.

In [ ]:
def find_email_from_hunter(company_domain, hr_name):
    print(f" Searching Hunter.io for {hr_name} at {company_domain}...")
    response = requests.get(
        "https://api.hunter.io/v2/email-finder",
        params={
            "domain": company_domain,
            "full_name": hr_name,
            "api_key": HUNTER_API_KEY
        }
    )
    email = response.json().get("data", {}).get("email", None)
    if email:
        print(f" Email found: {email}")
    else:
        print(" Could not find email automatically.")
    return email

print("Email finder ready!")


##  Step 6 — AI Email Writer
Uses Groq's LLaMA 3.3 70B to write personalized cold emails based on the LinkedIn post content.

In [ ]:
def load_email_examples():
    examples_file = r"C:\Users\Shourya\Desktop\cold_mail_bot\my_email_style.txt"
    try:
        with open(examples_file, 'r') as f:
            return f.read()
    except:
        return ""

def generate_cold_email(post_text, hr_name, your_profile):
    print(" Generating email with AI...")
    client = Groq(api_key=GROQ_API_KEY)
    my_examples = load_email_examples()

    prompt = f"""
You are helping someone write a cold email to an HR professional.

{"Study these example emails and copy their exact tone and style:" if my_examples else ""}
{my_examples if my_examples else ""}

The HR posted this on LinkedIn:
---
{post_text[:2000]}
---

HR's name: {hr_name if hr_name else "the HR Manager"}

About the sender:
{your_profile}

Rules:
- Match the style of the examples EXACTLY if provided
- Reference something specific from their LinkedIn post
- Keep it under 120 words
- Sound like a real person, not a bot
- No "I hope this email finds you well"
- End with one simple call to action

Only output the email body. Nothing else.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400
    )
    email_body = response.choices[0].message.content
    print(" Email generated!")
    print("\n--- EMAIL PREVIEW ---")
    print(email_body)
    print("---------------------\n")
    return email_body

def generate_subject_line(post_text):
    client = Groq(api_key=GROQ_API_KEY)
    prompt = f"""
Write ONE email subject line for a cold job application email.
Based on this LinkedIn post: {post_text[:500]}

Rules:
- Must mention the specific role or company name
- Under 8 words
- Direct and professional
- Examples: "Application for Data Analyst Role at Blinkit"
- No exclamation marks or question marks
- Only output the subject line, nothing else.
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=50
    )
    subject = response.choices[0].message.content.strip()
    print(f" Subject: {subject}")
    return subject

print(" AI writer ready!")


##  Step 7 — Gmail Sender
Sends email using Gmail API with OAuth — no password needed.

In [ ]:
SCOPES = [
    'https://www.googleapis.com/auth/gmail.send',
    'https://www.googleapis.com/auth/gmail.readonly'
]

def get_gmail_service():
    creds = None
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE, 'rb') as token:
            creds = pickle.load(token)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(TOKEN_FILE, 'wb') as token:
            pickle.dump(creds, token)
        print(" Google login saved!")
    return build('gmail', 'v1', credentials=creds)

def send_email_with_resume(to_email, subject, body, resume_path):
    print(f" Sending email to {to_email}...")
    service = get_gmail_service()
    msg = MIMEMultipart()
    msg['To'] = to_email
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))
    with open(resume_path, 'rb') as f:
        attachment = MIMEBase('application', 'octet-stream')
        attachment.set_payload(f.read())
        encoders.encode_base64(attachment)
        attachment.add_header('Content-Disposition', f'attachment; filename="{os.path.basename(resume_path)}"')
        msg.attach(attachment)
    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()
    service.users().messages().send(userId='me', body={'raw': raw}).execute()
    print(f" Email sent to {to_email}!")

print(" Gmail sender ready!")


##  Step 8 — Run the Bot
This is the main cell. Run this every time you want to send a cold email.

In [ ]:
print(" Starting Cold Mail Bot...\n")
print("━" * 50)

# Ask for inputs
LINKEDIN_POST_URL = input("🔗 Paste LinkedIn post URL: ").strip()
print("━" * 50)

print("\n📄 Enter full path to your resume PDF")
print('   Example: C:\\Users\\Shourya\\Desktop\\cold_mail_bot\\resume.pdf')
RESUME_PATH = input(" Resume path: ").strip().strip('"')
print("━" * 50)

if not os.path.exists(RESUME_PATH):
    print(f" Resume not found at: {RESUME_PATH}")
else:
    print(" Resume found!")

    # Step 1 — Scrape
    post_data = scrape_linkedin_post(LINKEDIN_POST_URL)
    post_text = post_data["full_text"]
    email_in_post = post_data["email_in_post"]

    # Step 2 — Get HR email
    print("━" * 50)
    HR_NAME = input(" HR full name: ").strip()
    COMPANY_DOMAIN = input(" Company domain (e.g. blinkit.com): ").strip()
    print("━" * 50)

    if email_in_post:
        hr_email = email_in_post
        print(f" Email found in post: {hr_email}")
    else:
        hr_email = find_email_from_hunter(COMPANY_DOMAIN, HR_NAME)

    if not hr_email:
        hr_email = input(" Enter HR email manually: ").strip()

    # Step 3 — Generate email
    email_body = generate_cold_email(post_text, HR_NAME, YOUR_PROFILE)
    subject_line = generate_subject_line(post_text)

    # Step 4 — Review and send
    print("\n" + "━" * 50)
    print(" FINAL REVIEW:")
    print(f"   To      : {hr_email}")
    print(f"   Subject : {subject_line}")
    print(f"   Resume  : {RESUME_PATH}")
    print("━" * 50)

    confirm = input("\n Type YES to send: ").strip()
    if confirm.upper() == "YES":
        send_email_with_resume(hr_email, subject_line, email_body, RESUME_PATH)
    else:
        print(" Cancelled.")
